# 32  Hiring, from Adzuna

**Two deliverables, and they are not the same thing.**

1. **Per company matching.** Which of our 1,493,972 companies are advertising jobs. Notebook
   07 matched 1 advert in 1,000 doing this, and this notebook measures that properly at a
   larger scale rather than repeating it.
2. **Market demand.** Open advert counts by industry and by region. This costs about 30 API
   calls, needs no matching at all, and is the part that actually works.

**Why the first one is hard, stated up front.** Adzuna is searched by keyword and location,
not by employer. There is no way to ask it "does company 08341276 have a job open". So we
pull the newest adverts we can afford, then try to match each employer name back to a
company. National advert feeds are dominated by recruitment agencies and large employers,
which is exactly the opposite of our SME universe. A low match rate is the expected result
and it belongs in the report as a measured limit of the source.

**One more limit worth stating.** Adzuna returns what is open **today**. There is no history,
so hiring can never appear on the timeline. It is a tile, dated the day it was pulled.

## 1. Setup and the quota probe

In [ ]:
from pathlib import Path
from getpass import getpass
from collections import defaultdict
import re, sys, time

import pandas as pd
import requests

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/Lloyds")
else:
    WORK_DIR = Path("..").resolve() / "data" / "processed"

OUT_DIR = WORK_DIR / "dashboard_pack"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MASTER = WORK_DIR / "company_master_gazette_thru_2026-07.csv"

SNAPSHOT_DATE = "2026-07-01"      # the window reference the rest of the pack uses
BASE = "https://api.adzuna.com/v1/api/jobs/gb"
UA = "LloydsBCB-MSc-project/1.0 (academic; contact via GitHub elyokerr)"

RESULTS_PER_PAGE = 50             # Adzuna maximum
PAGES = 120                       # raise or lower after reading the probe below

print("work dir:", WORK_DIR)

In [ ]:
ADZUNA_APP_ID = getpass("Adzuna app_id: ").strip()
ADZUNA_APP_KEY = getpass("Adzuna app_key: ").strip()
AUTH = {"app_id": ADZUNA_APP_ID, "app_key": ADZUNA_APP_KEY,
        "content-type": "application/json"}

# The probe. One call, before anything spends the quota, so the page budget below is set
# from what the account actually allows rather than from a guess.
r = requests.get(f"{BASE}/search/1", params={**AUTH, "results_per_page": 1},
                 headers={"User-Agent": UA}, timeout=60)
print("status:", r.status_code)

quota_headers = [k for k in r.headers
                 if any(w in k.lower() for w in ("rate", "limit", "quota", "remaining",
                                                 "retry", "x-ratelimit"))]
if quota_headers:
    print("\nwhat the API reports about your allowance:")
    for k in quota_headers:
        print(f"  {k}: {r.headers[k]}")
else:
    print("\nAdzuna returns no quota headers on this tier.")
    print("Published free allowance: 250 calls a day, 25 a minute.")
    print(f"At {RESULTS_PER_PAGE} results a page that is about "
          f"{250 * RESULTS_PER_PAGE:,} adverts a day at most.")

if r.status_code == 200:
    print(f"\ntotal open adverts in GB right now: {r.json().get('count', 0):,}")
    print(f"our {PAGES} pages will see about "
          f"{PAGES * RESULTS_PER_PAGE / max(r.json().get('count', 1), 1):.2%} of them")
else:
    # Do not raise. Print what Adzuna actually said, plus the shape of the
    # credentials, because a bare 401 is impossible to diagnose.
    print("REJECTED. What Adzuna sent back:")
    print(" ", r.text[:400])
    print()
    print("credential shape, the values themselves are never printed:")
    print(f"  app_id  length {len(ADZUNA_APP_ID):>3}   expected about 8")
    print(f"  app_key length {len(ADZUNA_APP_KEY):>3}   expected about 32")
    if len(ADZUNA_APP_ID) > len(ADZUNA_APP_KEY):
        print("  these look SWAPPED, re-run and paste them the other way round")
    print()
    print("the URL that was called, with the credentials masked:")
    print(" ", r.url.replace(ADZUNA_APP_ID, "APP_ID").replace(ADZUNA_APP_KEY, "APP_KEY"))

## 2. Market demand

This part needs no matching and no universe. It is 30 or so calls and it produces a real
result: where the hiring is, by industry and by area. On the dashboard it is context for a
company page, and in the report it is a paragraph that stands on its own.

In [ ]:
def as_json(r, what):
    """Adzuna answers with HTML or a bare error string when something is wrong, and
    calling .json() on that raises an unreadable JSONDecodeError. Show what came back
    instead, so the real problem is on screen."""
    if r.status_code != 200:
        print(f"  {what}: HTTP {r.status_code}")
        print(f"  body: {r.text[:200]}")
        return None
    try:
        return r.json()
    except ValueError:
        print(f"  {what}: status 200 but the body is not JSON")
        print(f"  content-type: {r.headers.get('content-type')}")
        print(f"  body: {r.text[:200]}")
        return None


def count_for(extra):
    r = requests.get(f"{BASE}/search/1", params={**AUTH, "results_per_page": 1, **extra},
                     headers={"User-Agent": UA}, timeout=60)
    j = as_json(r, f"count {extra}")
    return None if j is None else j.get("count", 0)


r = requests.get(f"{BASE}/categories", params=AUTH, headers={"User-Agent": UA}, timeout=60)
j = as_json(r, "categories")
cats = (j or {}).get("results", [])
if not cats:
    print("No categories returned. Fix that before running the rest of this notebook.")
    print("A 401 here with a working probe usually means the key is fine for /search")
    print("but the account is not activated yet. Wait a few minutes and retry.")
else:
    print(f"{len(cats)} categories")
rows = []
for c in cats:
    if c.get("tag") == "unknown":
        continue
    n = count_for({"category": c.get("tag")})
    if n is not None:
        rows.append({"industry": c.get("label"), "open_adverts": n})
    time.sleep(0.3)

if rows:
    cat_demand = pd.DataFrame(rows).sort_values("open_adverts", ascending=False).reset_index(drop=True)
    cat_demand["share"] = cat_demand["open_adverts"] / cat_demand["open_adverts"].sum()
    print(cat_demand.head(15).to_string(index=False))
    cat_demand.to_csv(OUT_DIR / "adzuna_demand_by_industry.csv", index=False)
else:
    cat_demand = pd.DataFrame(columns=["industry", "open_adverts"])
    print("nothing returned, see the message above")

In [ ]:
REGIONS = ["London", "Birmingham", "Manchester", "Leeds", "Glasgow", "Liverpool",
           "Bristol", "Sheffield", "Edinburgh", "Cardiff", "Belfast",
           "Newcastle upon Tyne", "Nottingham", "Leicester"]
rows = []
for rg in REGIONS:
    n = count_for({"where": rg})
    if n is not None:
        rows.append({"area": rg, "open_adverts": n})
    time.sleep(0.3)

if rows:
    reg_demand = pd.DataFrame(rows).sort_values("open_adverts", ascending=False).reset_index(drop=True)
    print(reg_demand.to_string(index=False))
    reg_demand.to_csv(OUT_DIR / "adzuna_demand_by_area.csv", index=False)
else:
    reg_demand = pd.DataFrame(columns=["area", "open_adverts"])
    print("nothing returned, see the message above")

## 3. Pull adverts

In [ ]:
def location_tokens(loc):
    """Adzuna's area is a hierarchy like [UK, Yorkshire, Leeds, Headingley], so the town can
    sit anywhere in it. Collect every level so a registered town can match any of them."""
    if not loc:
        return ""
    toks = set()
    for a in (loc.get("area") or [])[1:]:
        t = re.sub(r"[^A-Z ]", " ", str(a).upper()).strip()
        if t:
            toks.add(re.sub(r"\s+", " ", t))
    for x in str(loc.get("display_name", "")).split(","):
        t = re.sub(r"[^A-Z ]", " ", x.upper()).strip()
        if t:
            toks.add(re.sub(r"\s+", " ", t))
    return "|".join(sorted(toks))


cache = WORK_DIR / f"cache_adzuna_{PAGES}x{RESULTS_PER_PAGE}.parquet"

# A failed run must never leave a cache behind, or every later run reads the empty file
# and skips the pull. If an empty one is already there, throw it away.
if cache.exists() and len(pd.read_parquet(cache)) == 0:
    cache.unlink()
    print("removed an empty cache left by an earlier failed run")

if cache.exists():
    jobs = pd.read_parquet(cache)
    print(f"using the cached pull: {len(jobs):,} adverts")
else:
    rows, stopped = [], None
    for page in range(1, PAGES + 1):
        r = requests.get(f"{BASE}/search/{page}",
                         params={**AUTH, "results_per_page": RESULTS_PER_PAGE,
                                 "sort_by": "date"},
                         headers={"User-Agent": UA}, timeout=60)
        if r.status_code == 429:
            stopped = f"rate limited at page {page}"
            break
        if r.status_code != 200:
            stopped = f"HTTP {r.status_code} at page {page}"
            break
        body = as_json(r, f"page {page}")
        results = (body or {}).get("results", [])
        if not results:
            stopped = f"no more results at page {page}"
            break
        for j in results:
            rows.append({"employer": (j.get("company") or {}).get("display_name"),
                         "loc": location_tokens(j.get("location")),
                         "category": (j.get("category") or {}).get("label"),
                         "created": j.get("created"),
                         "title": j.get("title")})
        if page % 20 == 0:
            print(f"  page {page}, {len(rows):,} adverts")
        time.sleep(0.3)
    jobs = pd.DataFrame(rows)
    if len(jobs):
        jobs.to_parquet(cache, index=False)
    print(f"\npulled {len(jobs):,} adverts" + (f" ({stopped})" if stopped else ""))

if jobs.empty:
    jobs = pd.DataFrame(columns=["employer", "loc", "category", "created", "title"])
    raise SystemExit("No adverts were pulled. Fix the 401 in the probe cell first; "
                     "everything below needs adverts.")

print(f"  with an employer name: {jobs['employer'].notna().sum():,}")
print(f"  with a location      : {(jobs['loc'] != '').sum():,}")
print(f"  distinct employers   : {jobs['employer'].nunique():,}")
print("\nmost frequent advertisers, which shows what this feed is made of:")
print(jobs["employer"].value_counts().head(10).to_string())

## 4. Match to the universe

Name plus town, the same rule notebook 07 used. There is no fuzzy tier: a town is too coarse
to confirm a fuzzy name safely, and a wrong company on a page is worse than a missing one.

| Outcome | Confidence |
|---|---|
| name matches and the town is confirmed | 0.80 |
| name matches, no town on the advert, and only one company has that name | 0.50 |
| anything else | dropped |

In [ ]:
sys.path.insert(0, str(WORK_DIR))
from dashboard_export import normalise_name, write_source_pack

print("reading the universe ...")
uni = pd.read_csv(MASTER, dtype=str,
                  usecols=["CompanyNumber", "CompanyName", "RegAddress.PostTown"])
uni.columns = [c.strip() for c in uni.columns]
uni["nn"] = uni["CompanyName"].map(normalise_name)
uni["town"] = uni["RegAddress.PostTown"].fillna("").str.upper().str.strip()
uni = uni.dropna(subset=["nn"])
print(f"  {len(uni):,} companies")

by_name = defaultdict(list)
for cn, nn, tw in zip(uni["CompanyNumber"], uni["nn"], uni["town"]):
    by_name[nn].append((cn, tw))
print(f"  distinct normalised names: {len(by_name):,}")
del uni


def match_employer(name, loc):
    nn = normalise_name(name)
    if not nn:
        return None, 0.0, "no_name"
    cands = by_name.get(nn)
    if not cands:
        return None, 0.0, "name_not_found"
    towns = set(loc.split("|")) if loc else set()
    confirmed = [c for c, t in cands if t and t in towns]
    if confirmed:
        return confirmed[0], 0.8, "name_exact_town"
    if towns:
        return None, 0.0, "name_exact_town_mismatch"
    if len(cands) == 1:
        return cands[0][0], 0.5, "name_exact_unconfirmed"
    return None, 0.0, "name_exact_ambiguous"


out = [match_employer(r.employer, r.loc) for r in jobs.itertuples(index=False)]
jobs["cn"] = [o[0] for o in out]
jobs["confidence"] = [o[1] for o in out]
jobs["method"] = [o[2] for o in out]

m = jobs[jobs["cn"].notna()]
print(f"\nadverts matched: {len(m):,} of {len(jobs):,}  ({len(m)/max(len(jobs),1):.2%})")
print(f"companies       : {m['cn'].nunique():,}")
print("\nby outcome:")
print(jobs["method"].value_counts().to_string())

## 5. Write the pack

In [ ]:
if len(m):
    events = pd.DataFrame({
        "CompanyNumber": m["cn"].values,
        "event_date": pd.to_datetime(m["created"], errors="coerce", utc=True
                                     ).dt.date.astype(str).values,
        "event_type": "job_advert",
        "detail": m["title"].fillna("job advert").str.slice(0, 160).values,
        "value": 1.0,
        "url": "",
        "confidence": m["confidence"].values,
        "match_method": m["method"].values,
        "category": m["category"].values,
    })
    write_source_pack(events, prefix="hire", source="adzuna_hiring",
                      out_dir=OUT_DIR, snapshot_date=SNAPSHOT_DATE,
                      extra_cols=["category"])
    print("\nNote for the README: these adverts are open on the day of the pull.")
    print("hire_count_12m is not a 12 month history, it is the same adverts.")
    print("Label the tile with the pull date and do not put it on the timeline.")
else:
    print("No adverts matched. That is the result. Record the funnel in section 6.")

## 6. The funnel, for the report

In [ ]:
funnel = pd.DataFrame([
    {"stage": "adverts pulled", "n": len(jobs)},
    {"stage": "with an employer name", "n": int(jobs["employer"].notna().sum())},
    {"stage": "employer name found in our universe",
     "n": int((~jobs["method"].isin(["no_name", "name_not_found"])).sum())},
    {"stage": "matched to one company", "n": int(jobs["cn"].notna().sum())},
])
funnel["pct_of_pulled"] = (funnel["n"] / max(len(jobs), 1)).map("{:.2%}".format)
print(funnel.to_string(index=False))
funnel.to_csv(OUT_DIR / "adzuna_match_funnel.csv", index=False)

print("\nThe sentence for the report: Adzuna is searched by keyword and location, never by")
print("employer, so reaching a named SME means pulling the national feed and hoping it")
print("appears. The feed is dominated by recruitment agencies. This is a structural limit")
print("of the source, not a tuning problem.")